# Lecture 2: The Atoms Object

## Overview
**Questions**
- How can I describe a molecule or crystal using the `Atoms` class?
- How can I access and adjust `Atoms` information?
- What built-in help is available from ASE?

**Objectives**
- Create and visualise an `Atoms` object
- Return simple structural information from an `Atoms` object
- Adjust default properties of an `Atoms` object


## Molecules and materials are represented by the `Atoms` class

The `Atoms` class is the central data structure in ASE. It holds:
- **Chemical symbols** (or atomic numbers) of each atom
- **Positions** of each atom (in Ångströms)
- Optionally: the **unit cell**, **periodic boundary conditions**, **momenta**, **charges**, **magnetic moments**, ...

### Creating a simple molecule

We can define a diatomic molecule by providing lists of symbols and Cartesian positions:


In [ ]:
from ase import Atoms

# N2 molecule
d = 1.10  # N-N bond length in Angstrom
molecule = Atoms(['N', 'N'], positions=[(0., 0., 0.), (0., 0., d)])

# Shorthand: chemical formula string
molecule = Atoms('N2', positions=[(0., 0., 0.), (0., 0., d)])

print(f"Formula: {molecule.get_chemical_formula()}")
print(f"Number of atoms: {len(molecule)}")


### Visualising an `Atoms` object

We can use `ase.visualize.view` with the `nglview` backend to display structures inline in a Jupyter notebook.
> **Note:** If nglview is not installed, run `pip install nglview` and restart your kernel.


In [ ]:
from ase.visualize import view

# This will display an interactive 3D view in a Jupyter notebook.
# Left-click-and-drag to rotate; scroll to zoom.
# view(molecule, viewer='ngl')

# For non-interactive display we can use matplotlib:
import matplotlib.pyplot as plt
from ase.visualize.plot import plot_atoms

fig, ax = plt.subplots()
plot_atoms(molecule, ax, radii=0.5, rotation=('0x,0y,0z'))
ax.set_title('N₂ molecule')
ax.axis('off')
plt.tight_layout()
plt.show()


## Describing crystals: `cell` and `pbc`

Crystals are described by atomic positions inside a **periodic unit cell**. Two key keywords:
- `cell` — the unit cell vectors (3×3 matrix, or 3 lengths for cubic cells)
- `pbc` — periodic boundary conditions (True/False per direction)

### Example: Zinc blende GaN (a quantum optics workhorse)

GaN is a wide-bandgap semiconductor widely used in blue LEDs and increasingly explored as a host for quantum emitters. Here we construct the zinc blende polymorph.


In [ ]:
import numpy as np
from ase import Atoms

# Zinc blende GaN: a = 4.50 Angstrom
a = 4.50
gan = Atoms('Ga4N4',
            scaled_positions=[
                [0.00, 0.00, 0.00],  # Ga
                [0.00, 0.50, 0.50],  # Ga
                [0.50, 0.00, 0.50],  # Ga
                [0.50, 0.50, 0.00],  # Ga
                [0.25, 0.25, 0.25],  # N
                [0.25, 0.75, 0.75],  # N
                [0.75, 0.25, 0.75],  # N
                [0.75, 0.75, 0.25],  # N
            ],
            cell=[a, a, a],
            pbc=True)

print(f"Formula: {gan.get_chemical_formula()}")
print(f"Number of atoms: {len(gan)}")
print(f"Unit cell (Å):\n{gan.cell}")


In [ ]:
# Visualise the GaN crystal
from ase.visualize.plot import plot_atoms

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

plot_atoms(gan, axes[0], rotation=('10x,10y,0z'))
axes[0].set_title('GaN — [100] view')
axes[0].axis('off')

plot_atoms(gan, axes[1], rotation=('0x,90y,0z'))
axes[1].set_title('GaN — [010] view')
axes[1].axis('off')

plt.suptitle('Zinc blende GaN', fontsize=14)
plt.tight_layout()
plt.show()


## Getter methods

Once we have an `Atoms` object, we can retrieve information using **getter** methods:


In [ ]:
print("Positions (Å):")
print(gan.get_positions()[:4])  # first 4 atoms

print("\nChemical symbols:")
print(gan.get_chemical_symbols())

print("\nAtomic masses (u):")
print(gan.get_masses())

print("\nCell volume (Å³):", gan.get_volume())

# Compute Ga-N nearest-neighbour distance
from ase.geometry import get_distances
Ga_pos = gan.positions[0]   # first Ga atom
N_pos  = gan.positions[4]   # first N atom
dist, _ = get_distances([Ga_pos], [N_pos], cell=gan.cell, pbc=gan.pbc)
print(f"\nNearest Ga-N bond length: {dist[0,0]:.4f} Å")
print(f"  (ideal zinc blende: {a * np.sqrt(3)/4:.4f} Å)")


## Setter methods

We can modify an `Atoms` object after creation using **setter** methods:


In [ ]:
# Copy the structure and apply a small strain
import copy
strained = gan.copy()

# Biaxial strain in x-y plane (e.g. from epitaxial growth on a substrate)
cell = strained.get_cell()
strain = 0.02  # 2% tensile strain
cell[0] *= (1 + strain)
cell[1] *= (1 + strain)
strained.set_cell(cell, scale_atoms=True)

print(f"Original lattice constant: {gan.cell[0,0]:.4f} Å")
print(f"Strained lattice constant: {strained.cell[0,0]:.4f} Å")

# Magnetic moments — relevant for spin-based quantum systems
gan.set_initial_magnetic_moments([0.0] * len(gan))
print(f"\nInitial magnetic moments: {gan.get_initial_magnetic_moments()}")


## Key Points

- Molecules and materials are represented by the `Atoms` class
- `ase.visualize.view` or `plot_atoms` can visualise an `Atoms` object
- Crystals require `cell` (unit cell) and `pbc` (periodic boundary conditions)
- Getter methods (`get_positions()`, `get_masses()`, ...) retrieve information
- Setter methods (`set_cell()`, `set_initial_magnetic_moments()`, ...) modify the object

## Exercise 2.1

Construct a wurtzite GaN unit cell (the more common hexagonal polymorph). It has:
- Lattice parameters: a = 3.19 Å, c = 5.19 Å
- Space group P6₃mc; Wyckoff positions 2b for both Ga and N
- Scaled positions: Ga at (1/3, 2/3, 0) and (2/3, 1/3, 1/2); N at (1/3, 2/3, 3/8) and (2/3, 1/3, 7/8)

Hint: for a hexagonal cell you need the full 3×3 matrix. Use `ase.build.bulk('GaN', crystalstructure='wurtzite', a=3.19, c=5.19)` to check your answer.

## Exercise 2.2

The NV centre in diamond consists of a nitrogen substitutional adjacent to a carbon vacancy. Build a diamond cubic unit cell (a = 3.57 Å, 8 atoms) and print the nearest-neighbour C-C bond length. Compare to the known value of 1.54 Å.
